# CareLoop — Week 1 Overview

Exploratory orientation for the CareLoop prototype. This notebook is markdown-only
for Week 1; executable cells are added as data/rule/optimiser work lands.

## Scope

Explainable decision support: turn a **clinician-approved nutrient envelope**
(T2D, hypertension, CKD stage 3–4) into realistic Singapore meal plans, with
traceability from clinical guideline → fired rule → approved constraint → plated meal.

## The contract: `NutrientEnvelope`

Defined once in `shared/envelope_schema.py` (Pydantic v2) and
`shared/envelope_schema.json` (JSON Schema). It is the boundary between every module.

- `status` ∈ {`PROPOSED`, `APPROVED`, `REJECTED`, `EDITED`}
- `targets`: per-nutrient target/min/max constraints
- `exclusions`: hard ingredient/category/pattern exclusions with citations
- `fired_rules`: the explainability trace (rule id, priority, citation)
- `priority_basis`: human-readable conflict priority

**Approval gate invariant:** only `APPROVED` envelopes may be consumed downstream
(optimiser, RAG chat, patient views). Enforced in the API, not by convention.

## Clinical priority hierarchy

`CKD > hypertension > T2D` on conflicts. Concrete cases:

| Case | Outcome |
|---|---|
| CKD + HTN | CKD sodium ceiling (KDIGO <2000 mg/d) wins over DASH |
| CKD + normal K+ | No blanket potassium restriction despite DASH's K ~4700 mg/d |
| Hyperkalemia (K+ high) | Potassium restriction rule fires |

## Data sources

- HPB SG FoodID (curated subset, ~150–300 local dishes)
- USDA FoodData Central for K/P gaps (raw ingredients)
- Synthetic patients (diagnoses, eGFR, K+, PO4, weight, meds, longitudinal labs)

## Stack decisions

| Area | Choice |
|---|---|
| Rules | `clipspy` (CLIPS 6.4x) |
| Optimisation | `pymoo` NSGA-II + baselines |
| Graph | SQLite + NetworkX (Week 1 gate) |
| RAG | ChromaDB + sentence-transformers + LLM (Ollama/OpenAI/Gemini) |
| API | FastAPI + Pydantic v2 |

## Week 1 exit criteria

- [ ] Envelope schema frozen
- [ ] ≥30 rules specced
- [ ] ≥150 dishes with energy/protein/Na/K/P/carbs
- [ ] ≥10 synthetic patients
- [ ] API serves stub endpoints
